In [2]:
import sys, os
from pathlib import Path

cwd  = Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)


In [3]:
from src.multimodal import MultimodalDataset

ds = MultimodalDataset.load('dataset/preprocessed_dataset.npz', batch_size=16)
print(ds)


MultimodalDataset(n_subjects=80, sh_order=10, hrtf_shape=(121, 129, 2), train=57 / val=11 / test=12)


In [ ]:
import numpy as np
import tensorflow as tf
from src.models import EarEncoder, HRTFEncoder, ContrastiveModel

#Charge la structure du modèle 

enc_ear  = EarEncoder(embedding_dim=128)
enc_hrtf = HRTFEncoder(n_sh=121, n_freqs=129, embedding_dim=128)
model    = ContrastiveModel(enc_ear, enc_hrtf, temperature=0.07)

# Build avec un batch factice
dummy = {
    'ear_left':  np.zeros((1, 224, 224, 3), dtype='float32'),
    'ear_right': np.zeros((1, 224, 224, 3), dtype='float32'),
    'hrtf':      np.zeros((1, 121, 129, 2), dtype='float32'),
}
model(dummy, training=False)

# Charger les poids — adapte le chemin selon ton expérience
weights_path = sorted(Path('checkpoints').glob('*/phase*_best.weights.h5'))[-1]
model.load_weights(str(weights_path))
print(f'Poids chargés : {weights_path}')


Poids chargés : checkpoints\exp_temp007_bs8_2d83030a\phase1_best.weights.h5


In [5]:
from src.inference import HRTFPredictor

predictor = HRTFPredictor.build(model, ds, split='train', sh_order=10)
predictor.save_gallery('checkpoints/gallery.npz')
print(predictor)


  Gallery construite — 57 sujets (split=train)
  Gallery sauvegardée → checkpoints/gallery.npz
HRTFPredictor(gallery=Gallery(n=57, D=128), sh_order=10)


In [7]:
result = predictor.predict(
    'dataset/Database-Master_V2-1/H10/H10_Scans/H10_(L).png',
    'dataset/Database-Master_V2-1/H10/H10_Scans/H10_(R).png',
)

print(f"Sujet matché  : {result['subject_id']}")
print(f"Similarité    : {result['similarity']:.3f}")
print(f"HRTF shape    : {result['hrtf_coeffs'].shape}")


Sujet matché  : H14
Similarité    : -0.020
HRTF shape    : (121, 129, 2)
